# SQL Intermediate 2

**Estimated time:** ~9 hours total (about 4 hours reading and running this guide + ~5 hours on
`sql-intermediate-2-exercises.ipynb`).

`sql-intermediate` gave you joins, subqueries, CTEs and the core window functions. That is enough to answer
most business questions correctly. It is not yet enough for the questions that show up once you have been
doing this for a while: *which two products keep selling together? how deep does this org chart go, and can I
flatten it without a recursive CTE? why did my running total just count the same day's orders twice?*

This level is a bridge. It reuses everything from `sql-intermediate` — CTEs, window functions, correlated
subqueries — and points them at problems that are still intermediate in difficulty but do not come up until
you have written a few dozen real reports. `sql-advanced` starts immediately after this with a different kind
of question: not "is it correct" but "is it fast", which needs `EXPLAIN` and indexes to answer. This level
stays entirely in the world of getting the right rows out.

## Who This Is For

You finished `sql-intermediate` — you can write a CTE chain, use `ROW_NUMBER`/`RANK`/`LAG`/`LEAD`, and you
already know why a join before an aggregate doubles a total. A mentor or a mentee said the jump from
`sql-intermediate` straight to `sql-advanced` skipped things that come up constantly in practice. This is
those things.

## What You Will Learn

- Deduplicating rows that represent the same real-world entity, using data that actually has duplicates in it
- Self-joins used for **association** rather than hierarchy — which products get bought together
- Flattening a shallow hierarchy by hand, as the technique to reach for before a hierarchy needs a recursive CTE
- Unpivoting — turning columns into rows, the reverse of the cross-tab `sql-intermediate` built
- `FIRST_VALUE` and `LAST_VALUE`, and the frame `LAST_VALUE` needs that nothing warns you about
- `RANGE` versus `ROWS`, and the tied rows that make the default frame quietly wrong
- `PERCENT_RANK` and `CUME_DIST` for percentiles finer than `NTILE` can cut
- The same question answered three ways — correlated subquery, join, window function — and how to choose
- Building a date spine with a hand-written list of numbers, before `sql-advanced` shows you the recursive way
- Turning a nested subquery nobody wants to touch into a CTE chain, without changing the answer
- Three classic interview patterns: second-highest without `LIMIT`/`OFFSET`, the longest gap between two
  events, and a year-over-year comparison

## How to Use This Guide

Run every cell with **Shift + Enter**, in order. The setup cell rebuilds the same shop database from
`sql-basics` and `sql-intermediate` — nothing about the schema has changed, only the questions being asked
of it.

Before you run a query, guess what one row of the result means and roughly how many rows will come back. Most
of this level's lessons are places where that guess is wrong the first time, on purpose.

**Practice:** `sql-intermediate-2-exercises.ipynb` follows this guide section by section.


## 1. Setup — The Same Shop, New Questions

Same seven tables, same `../assets/sql/` CSVs, same imperfections. Two of them were mentioned but not used
much in `sql-intermediate`, and this level spends real time on them: the two customers who signed up twice
with the same email, and the fact that `employees.manager_id` only goes three levels deep.

Run the cell to build it.


In [2]:
import sqlite3
import pandas as pd

SQL = "assets/sql"          # the bundled schema and CSV files
TABLES = ["categories", "customers", "employees", "products",
          "orders", "order_items", "payments"]

con = sqlite3.connect(":memory:")         # the database lives in RAM -- nothing to clean up

with open(f"{SQL}/schema.sql") as f:
    con.executescript(f.read())           # creates the seven empty tables

con.execute("PRAGMA foreign_keys = ON")   # from here on, SQLite enforces the foreign keys

for table in TABLES:
    pd.read_csv(f"{SQL}/{table}.csv").to_sql(table, con, if_exists="append", index=False)
con.commit()


def q(sql):
    """Run a SELECT and hand the result back as a pandas DataFrame."""
    return pd.read_sql_query(sql, con)


def run(sql):
    """Run statements that change data or structure: CREATE, INSERT, UPDATE, DELETE."""
    con.executescript(sql)
    con.commit()


pd.set_option("display.width", 120)
pd.set_option("display.max_rows", 25)

for table in TABLES:
    print(f"{table:12s} {q(f'SELECT COUNT(*) AS n FROM {table}')['n'][0]:>4} rows")


categories      8 rows
customers      60 rows
employees      15 rows
products       40 rows
orders        300 rows
order_items   673 rows
payments      248 rows


**Step by step:**

1. This is the identical setup cell from `sql-intermediate` — same database, same in-memory build, same `q`
   and `run` helpers. If you have that notebook's habits, they all carry over.
2. Nothing here has changed since the basics level: 60 customers, 15 employees, 40 products, 300 orders, 673
   order lines, 248 payments, 8 categories.
3. Two rows in `customers` share an email address (`neha.reddy@example.com` and `tarun.khan@example.com`) —
   section 2 uses them for real.
4. `employees.manager_id` chains exactly three deep: one founder, three managers, eleven reps and agents.
   Section 4 flattens that by hand.
5. Everything else about the data — the imperfections, the revenue formula, the date range — is exactly what
   `sql-basics` and `sql-intermediate` documented.


## 2. Deduplication — Keeping One Row per Duplicate Entity

Every join and `GROUP BY` you have written so far assumed one row per real-world thing. That assumption breaks
the moment the same person signs up twice, the same product gets entered under two ids, or two systems get
merged and nobody deduplicated first. This is the shape of problem you meet the first time you touch a real
CRM export.

`customers` has exactly this: two people, two rows each, same email, different `customer_id` and
`signup_date`. `GROUP BY email HAVING COUNT(*) > 1` finds them — `sql-intermediate` section 20 already used
that as a data quality check. This section goes one step further and actually picks which row to keep.

The standard shape: rank the duplicates with `ROW_NUMBER() OVER (PARTITION BY <the thing that should be
unique> ORDER BY <the tiebreaker that decides which one survives>)`, then keep only `rn = 1`.


In [2]:
q("""
SELECT customer_id, name, email, city, signup_date,
       ROW_NUMBER() OVER (PARTITION BY email ORDER BY signup_date, customer_id) AS rn
FROM customers
WHERE email IN (SELECT email FROM customers GROUP BY email HAVING COUNT(*) > 1)
ORDER BY email, rn
""")


,customer_id,name,email,city,signup_date,rn
0,59,Neha Reddy,neha.reddy@example.com,Mumbai,2023-02-18,1
1,12,Neha Reddy,neha.reddy@example.com,Mumbai,2023-05-20,2
2,28,Tarun Khan,tarun.khan@example.com,Bengaluru,2023-03-29,1
3,60,Tarun Khan,tarun.khan@example.com,Bengaluru,2023-09-27,2


In [3]:
deduped = q("""
WITH ranked AS (
    SELECT customer_id, email,
           ROW_NUMBER() OVER (PARTITION BY email ORDER BY signup_date, customer_id) AS rn
    FROM customers
)
SELECT COUNT(*) AS n FROM ranked WHERE rn = 1
""")["n"][0]

print("customers total :", q("SELECT COUNT(*) AS n FROM customers")["n"][0])
print("after dedup     :", deduped)


customers total : 60
after dedup     : 58


**Step by step:**

1. The `PARTITION BY email` groups the two pairs of duplicates together; `ORDER BY signup_date, customer_id`
   decides survivorship — here, the **earliest** signup wins, with `customer_id` as a tiebreaker so the answer
   is reproducible.
2. `rn = 1` picks that earliest row in each partition; every other email, which has no duplicate, also gets
   `rn = 1` because a partition of one row is still ranked.
3. Wrapping the ranked query in an outer `SELECT ... WHERE rn = 1` is exactly the top-N-per-group shape from
   `sql-intermediate` section 13 — deduplication is that pattern with `N = 1`.
4. 60 customers become 58 once the two duplicate rows are dropped. Every downstream count that should be "one
   row per real customer" — a mailing list, a segmentation report — needs this step first, or it double-counts
   two people.
5. Which row should survive is a judgement call, not a SQL question: earliest signup, most complete row (the
   one with a `city` filled in), or most recent activity are all defensible. Decide before you write the
   `ORDER BY`, because the query will not tell you if you picked the wrong side.


## 3. Self-Joins for Association — What Gets Bought Together

`sql-intermediate` used a self-join for one thing: walking a hierarchy, `employees` joined to itself through
`manager_id`. A self-join has a second, unrelated use — finding pairs of rows that share a group, which is the
basis of "customers who bought X also bought Y".

Join `order_items` to itself on `order_id`, and every pair of *different* products that showed up on the same
order comes out as one row. `i2.product_id > i1.product_id` in the join condition (not the `WHERE`) does two
things at once: it excludes a product pairing with itself, and it keeps each pair once instead of twice
(`(A, B)` and `(B, A)` would otherwise both appear).


In [4]:
q("""
WITH pairs AS (
    SELECT i1.product_id AS p1, i2.product_id AS p2, i1.order_id
    FROM order_items i1
    JOIN order_items i2 ON i2.order_id = i1.order_id AND i2.product_id > i1.product_id
)
SELECT p1.name AS product_a, p2.name AS product_b, COUNT(*) AS times_together
FROM pairs
JOIN products p1 ON p1.product_id = pairs.p1
JOIN products p2 ON p2.product_id = pairs.p2
GROUP BY pairs.p1, pairs.p2
ORDER BY times_together DESC, product_a, product_b
LIMIT 8
""")


,product_a,product_b,times_together
0,Braid USB-C Cable 1m,Anchor 100W Charger,4
1,Braid USB-C Cable 1m,Clarity 24 Monitor,4
2,Braid USB-C Cable 1m,Vault 4TB HDD,4
3,Clarity 32 4K Monitor,Vault 2TB SSD,4
4,Echo Buds Pro,Rumble Bluetooth Speaker,4
5,Frame Compact Camera,Frame 50mm Lens,4
6,Quiet Desk Mic,Vault 4TB HDD,4
7,Vault 1TB SSD,Vault 4TB HDD,4


In [5]:
q("""
WITH pairs AS (
    SELECT i1.product_id AS p1, i2.product_id AS p2
    FROM order_items i1
    JOIN order_items i2 ON i2.order_id = i1.order_id AND i2.product_id <> i1.product_id
),
counted AS (
    SELECT p1, p2, COUNT(*) AS times_together FROM pairs GROUP BY p1, p2
),
ranked AS (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY p1 ORDER BY times_together DESC, p2) AS rn
    FROM counted
)
SELECT pr.name AS product, pa.name AS best_partner, r.times_together
FROM ranked r
JOIN products pr ON pr.product_id = r.p1
JOIN products pa ON pa.product_id = r.p2
WHERE r.rn = 1
ORDER BY r.times_together DESC, pr.name
LIMIT 5
""")


,product,best_partner,times_together
0,Anchor 100W Charger,Braid USB-C Cable 1m,4
1,Braid USB-C Cable 1m,Anchor 100W Charger,4
2,Clarity 24 Monitor,Braid USB-C Cable 1m,4
3,Clarity 32 4K Monitor,Vault 2TB SSD,4
4,Echo Buds Pro,Rumble Bluetooth Speaker,4


**Step by step:**

1. `pairs` is the association step: same `order_id`, two different `order_items` rows, `p2 > p1` so each
   unordered pair is counted once. Joining `order_items` back to itself is what makes "things that co-occur"
   answerable in SQL at all — there is no other way to compare two rows to each other without a self-join.
2. `GROUP BY pairs.p1, pairs.p2` then counts how often each pair occurred, which is the first query above.
3. The second query is the top-N-per-group shape again, this time with `p2 <> p1` (both directions, so every
   product gets to be `p1` at least once) instead of `p2 > p1`: rank each product's partners, keep `rn = 1`,
   and you have "the one product most often bought with this one".
4. The two self-joins in this section are doing different jobs from the hierarchy self-join: there, `m` was
   semantically "the other row above this one"; here, `i2` is "some other row in the same order". Same SQL
   mechanism, unrelated question.
5. This is the whole idea behind "customers who bought this also bought" features — no machine learning
   involved, just a self-join and a count.


## 4. Flattening a Shallow Hierarchy Without Recursion

`employees` is a hierarchy — `sql-intermediate` walked one level of it with a self-join to find each person's
manager. This company's hierarchy is only three levels deep: founder, managers, everyone else. When you know
the depth in advance and it is small, you do not need a recursive CTE (that is `sql-advanced` section 8) — you
chain the same self-join once per level.


In [6]:
q("""
SELECT e.employee_id, e.name AS employee, e.role,
       m.name AS manager, m.role AS manager_role,
       g.name AS director, g.role AS director_role
FROM employees e
LEFT JOIN employees m ON m.employee_id = e.manager_id
LEFT JOIN employees g ON g.employee_id = m.manager_id
ORDER BY e.employee_id
LIMIT 6
""")


,employee_id,employee,role,manager,manager_role,director,director_role
0,1,Radhika Menon,Founder,None,None,None,None
1,2,Vikram Nair,Sales Manager,Radhika Menon,Founder,None,None
2,3,Sunita Rao,Sales Manager,Radhika Menon,Founder,None,None
3,4,Imran Sheikh,Support Manager,Radhika Menon,Founder,None,None
4,5,Arjun Pillai,Sales Rep,Vikram Nair,Sales Manager,Radhika Menon,Founder
5,6,Kavya Krishnan,Sales Rep,Vikram Nair,Sales Manager,Radhika Menon,Founder


In [7]:
q("""
SELECT e.employee_id, e.name,
       CASE WHEN e.manager_id IS NULL THEN 1
            WHEN m.manager_id IS NULL THEN 2
            ELSE 3 END AS level
FROM employees e
LEFT JOIN employees m ON m.employee_id = e.manager_id
ORDER BY level, e.employee_id
""")


,employee_id,name,level
0,1,Radhika Menon,1
1,2,Vikram Nair,2
2,3,Sunita Rao,2
3,4,Imran Sheikh,2
4,5,Arjun Pillai,3
5,6,Kavya Krishnan,3
6,7,Devendra Joshi,3
7,8,Priya Balan,3
8,9,Nikhil Verma,3
9,10,Farah Qureshi,3


**Step by step:**

1. Every `LEFT JOIN` in the chain walks up one more level: `e` to `m` is "my manager", `m` to `g` is "my
   manager's manager". Three self-joins would reach a fourth level, and so on.
2. All three joins have to be `LEFT`, not `INNER`. The founder has no manager, so `m` is `NULL` for that row;
   an `INNER JOIN` would drop the founder from the report entirely, the same lesson `sql-intermediate` section
   2 taught with one level.
3. `director` and `director_role` are `NULL` for the founder and the three managers — there is nobody two
   levels above them — and filled in only for the eleven reps and agents at the bottom.
4. The second query assigns each employee a `level` number by checking, level by level, where the chain of
   `manager_id`s stops being `NULL`. It only works because the depth (3) is known and fixed in the query text.
5. The moment the depth is unknown, or a fourth level could be added without changing the query — a real org
   chart, a category tree with subcategories of subcategories — chained self-joins stop working and you need
   the recursive CTE from `sql-advanced`. Knowing when to reach for which is the actual skill; the SQL for the
   fixed-depth case is what you just wrote.


## 5. Unpivoting — Turning Columns Into Rows

`sql-intermediate` section 17 built a cross-tab: one row per month, one column per channel. That direction —
rows into columns — is called **pivoting**. The reverse, columns into rows, is **unpivoting**, and it comes up
just as often: a wide table with one column per metric is awkward to filter, chart, or load into a system that
expects "one fact per row".

SQL has no `UNPIVOT` keyword in SQLite (`UNPIVOT` exists in some other engines). The portable way is
`UNION ALL`, one `SELECT` per column being unpivoted.


In [8]:
q("""
SELECT product_id, name, 'price' AS metric, price AS value FROM products
UNION ALL
SELECT product_id, name, 'cost',  cost  FROM products
UNION ALL
SELECT product_id, name, 'stock', stock FROM products
ORDER BY product_id, metric
LIMIT 6
""")


,product_id,name,metric,value
0,1,Aster 14 Laptop,cost,51000.0
1,1,Aster 14 Laptop,price,62000.0
2,1,Aster 14 Laptop,stock,24.0
3,2,Aster 15 Pro Laptop,cost,78000.0
4,2,Aster 15 Pro Laptop,price,94000.0
5,2,Aster 15 Pro Laptop,stock,11.0


In [9]:
q("""
WITH wide AS (
    SELECT strftime('%Y-%m', o.order_date) AS month,
           SUM(CASE WHEN o.channel = 'web'   THEN i.quantity * i.unit_price * (1 - i.discount) ELSE 0 END) AS web,
           SUM(CASE WHEN o.channel = 'app'   THEN i.quantity * i.unit_price * (1 - i.discount) ELSE 0 END) AS app,
           SUM(CASE WHEN o.channel = 'store' THEN i.quantity * i.unit_price * (1 - i.discount) ELSE 0 END) AS store,
           SUM(CASE WHEN o.channel = 'phone' THEN i.quantity * i.unit_price * (1 - i.discount) ELSE 0 END) AS phone
    FROM orders o
    JOIN order_items i ON i.order_id = o.order_id
    WHERE o.status != 'cancelled' AND strftime('%Y', o.order_date) = '2024'
    GROUP BY month
)
SELECT month, 'web' AS channel, ROUND(web, 2) AS revenue FROM wide
UNION ALL SELECT month, 'app',   ROUND(app, 2)   FROM wide
UNION ALL SELECT month, 'store', ROUND(store, 2) FROM wide
UNION ALL SELECT month, 'phone', ROUND(phone, 2) FROM wide
ORDER BY month, channel
LIMIT 8
""")


,month,channel,revenue
0,2024-01,app,3240.0
1,2024-01,phone,13320.0
2,2024-01,store,264300.0
3,2024-01,web,133270.0
4,2024-02,app,127230.0
5,2024-02,phone,0.0
6,2024-02,store,125720.0
7,2024-02,web,63000.0


**Step by step:**

1. Each `SELECT` in the first query contributes one metric as its own row: same `product_id` and `name`, a
   literal string naming the metric, and the column's value renamed to a shared `value` column. `UNION ALL`
   stacks the three, giving three rows per product instead of one row with three columns.
2. Every branch of a `UNION ALL` must return the same number of columns with compatible types, in the same
   order — that discipline is why the literal `'price'`/`'cost'`/`'stock'` has to sit in the same position in
   every branch.
3. The second query unpivots a cross-tab built the `sql-intermediate` way — `web`/`app`/`store`/`phone` as
   columns — back into `month, channel, revenue` rows. This is the shape most charting libraries and BI tools
   actually want; the wide table was for a human to read, the long table is for a machine to plot.
4. Notice February's `phone` revenue is `0.0` in the long form — the `CASE WHEN ... ELSE 0` in the wide query
   guaranteed every channel has a row for every month, even a month with no phone orders at all. Losing that
   zero would make the chart look like the month is missing data rather than a channel with no sales.
5. `UNION ALL`, not `UNION`: there is no reason to deduplicate rows that are already guaranteed distinct by
   `product_id`/`metric` or `month`/`channel`, and `UNION` would pay the cost of checking anyway.


## 6. FIRST_VALUE and LAST_VALUE — the Frame `LAST_VALUE` Needs That Nothing Warns You About

`ROW_NUMBER`, `RANK` and `LAG`/`LEAD` cover most window-function needs, but two questions are cleaner with
`FIRST_VALUE` and `LAST_VALUE`: "what was the cheapest product in this category" on every row of that
category, or "what channel did this customer first/last order through" on every row of their orders.

`FIRST_VALUE` just works with the default frame. `LAST_VALUE` almost never does — and the failure is silent.


In [10]:
q("""
SELECT name, category_id, price,
       FIRST_VALUE(name) OVER (PARTITION BY category_id ORDER BY price ASC)  AS cheapest_in_cat,
       FIRST_VALUE(name) OVER (PARTITION BY category_id ORDER BY price DESC) AS priciest_in_cat
FROM products
ORDER BY category_id, price
LIMIT 5
""")


,name,category_id,price,cheapest_in_cat,priciest_in_cat
0,Vega Book 13,1,54000.0,Vega Book 13,Vega Book 16 Studio
1,Aster 14 Laptop,1,62000.0,Vega Book 13,Vega Book 16 Studio
2,Aster 15 Pro Laptop,1,94000.0,Vega Book 13,Vega Book 16 Studio
3,Nimbus Air Laptop,1,118000.0,Vega Book 13,Vega Book 16 Studio
4,Vega Book 16 Studio,1,142000.0,Vega Book 13,Vega Book 16 Studio


In [11]:
# the trap: LAST_VALUE with no explicit frame
wrong = q("""
WITH ord AS (
    SELECT customer_id, order_date, channel,
           LAST_VALUE(channel) OVER (PARTITION BY customer_id ORDER BY order_date, order_id) AS last_channel,
           ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date, order_id) AS rn
    FROM orders WHERE status != 'cancelled'
)
SELECT customer_id, last_channel FROM ord WHERE rn = 1
ORDER BY customer_id LIMIT 3
""")

# the fix: an explicit frame that spans the whole partition
right = q("""
WITH ord AS (
    SELECT customer_id, order_date, channel,
           FIRST_VALUE(channel) OVER (PARTITION BY customer_id ORDER BY order_date, order_id) AS first_channel,
           LAST_VALUE(channel) OVER (PARTITION BY customer_id ORDER BY order_date, order_id
                                      RANGE BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) AS last_channel,
           ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date, order_id) AS rn
    FROM orders WHERE status != 'cancelled'
)
SELECT customer_id, first_channel, last_channel FROM ord WHERE rn = 1
ORDER BY customer_id LIMIT 3
""")

print("LAST_VALUE, default frame (wrong):")
print(wrong)
print()
print("LAST_VALUE, explicit whole-partition frame (right):")
print(right)


LAST_VALUE, default frame (wrong):
   customer_id last_channel
0            1          web
1            2          web
2            3          app

LAST_VALUE, explicit whole-partition frame (right):
   customer_id first_channel last_channel
0            1           web          app
1            2           web          web
2            3           app          web


**Step by step:**

1. The first query needs no frame work: `FIRST_VALUE(name) OVER (PARTITION BY category_id ORDER BY price ASC)`
   puts "the cheapest product in this category" on every row of that category, and the same trick with
   `DESC` gives the priciest. `sql-intermediate` section 8's correlated subquery could answer one of these;
   this answers both in a single pass.
2. `LAST_VALUE`'s default frame is `RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` — the same default every
   other window function in this course has used. For `FIRST_VALUE` that default is harmless, because the
   first row is always in the frame no matter where the frame ends. For `LAST_VALUE` it is fatal: on customer
   1's first row, the frame is just that one row, so "the last value in the frame" is customer 1's *first*
   channel (`web`), not their actual last one (`app`).
3. The fix is the same explicit frame `sql-intermediate` section 15 used to make a grand total ignore
   `ORDER BY`: `ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING` (or `RANGE`, here — see section 7 for
   when the two differ) forces the frame to cover the whole partition regardless of which row you are
   computing it for.
4. With that frame, `last_channel` on customer 1's row now correctly reads `app`, matching the last row of
   their own order history shown above.
5. `LAST_VALUE` without an explicit frame is one of the most common silent window-function bugs there is,
   because the query runs, returns a value, and that value is even *plausible* — it is just the wrong row's
   value.


## 7. RANGE versus ROWS — the Frame Default That Duplicates Ties

`sql-intermediate` wrote frames as `ROWS BETWEEN ...` throughout. SQL actually offers two frame units,
`ROWS` and `RANGE`, and they agree everywhere — except when the `ORDER BY` has ties. Three orders were placed
on `2024-12-31`. That is enough to show the difference.


In [12]:
q("""
SELECT order_id, order_date, status FROM orders WHERE order_date = '2024-12-31' ORDER BY order_id
""")


,order_id,order_date,status
0,277,2024-12-31,placed
1,279,2024-12-31,delivered
2,285,2024-12-31,shipped


In [13]:
q("""
WITH order_rev AS (
    SELECT o.order_id, o.order_date, SUM(i.quantity * i.unit_price * (1 - i.discount)) AS revenue
    FROM orders o JOIN order_items i ON i.order_id = o.order_id
    WHERE o.status != 'cancelled' AND o.order_date BETWEEN '2024-12-28' AND '2024-12-31'
    GROUP BY o.order_id
)
SELECT order_id, order_date, ROUND(revenue, 2) AS revenue,
       ROUND(SUM(revenue) OVER (ORDER BY order_date
             ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW), 2)  AS running_rows,
       ROUND(SUM(revenue) OVER (ORDER BY order_date
             RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW), 2) AS running_range
FROM order_rev
ORDER BY order_date, order_id
""")


,order_id,order_date,revenue,running_rows,running_range
0,281,2024-12-28,34000.0,34000.0,34000.0
1,283,2024-12-29,28300.0,62300.0,62300.0
2,277,2024-12-31,38000.0,100300.0,178575.0
3,279,2024-12-31,22525.0,122825.0,178575.0
4,285,2024-12-31,55750.0,178575.0,178575.0


**Step by step:**

1. Orders 277, 279 and 285 all fall on `2024-12-31` — three rows tied on the column the window is ordered by.
2. `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` counts **physical rows**: row by row, regardless of
   ties, so `running_rows` climbs by exactly one order's revenue at a time — this is the running total
   `sql-intermediate` section 15 taught, and it behaves exactly as expected.
3. `RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` counts by **value**: every row that ties on `order_date`
   is treated as a single peer group, all included or all excluded together. All three `2024-12-31` rows land
   in the same peer group, so all three get the *same* `running_range` — the total including all of them, as if
   the frame had already reached the end of the group before computing any one of their values.
4. For a running total you almost always want `ROWS`, because "how much had come in strictly up to and
   including this order" should differ order by order, even same-day orders. `RANGE` is right when the
   question is genuinely about the value tied on, not the row — a percentile cutoff, for instance, where every
   row with the same score should get the same answer.
5. This is exactly the trap `LAST_VALUE` fell into in section 6: `RANGE` was used there deliberately, to make
   every row in a partition see the same whole-partition frame. Whether you want `ROWS` or `RANGE` depends on
   whether ties in the `ORDER BY` should be treated as one group or kept apart — decide that on purpose, because
   the default answers it for you.


## 8. PERCENT_RANK and CUME_DIST — Percentiles Beyond NTILE

`sql-intermediate` section 16 used `NTILE(4)` to split customers into quartiles. `NTILE` only cuts a
population into however many equal-sized buckets you ask for. `PERCENT_RANK` and `CUME_DIST` give every row
its own precise percentile instead of a bucket number.


In [14]:
q("""
WITH cust AS (
    SELECT o.customer_id, SUM(i.quantity * i.unit_price * (1 - i.discount)) AS ltv
    FROM orders o JOIN order_items i ON i.order_id = o.order_id
    WHERE o.status != 'cancelled'
    GROUP BY o.customer_id
)
SELECT customer_id, ROUND(ltv, 2) AS ltv,
       ROUND(PERCENT_RANK() OVER (ORDER BY ltv), 4) AS pct_rank,
       ROUND(CUME_DIST()   OVER (ORDER BY ltv), 4) AS cume_dist
FROM cust
ORDER BY ltv DESC
LIMIT 5
""")


,customer_id,ltv,pct_rank,cume_dist
0,6,2835360.0,1.00,1.0000
1,59,1539182.5,0.98,0.9804
2,34,1257470.0,0.96,0.9608
3,19,1002250.0,0.94,0.9412
4,38,826850.0,0.92,0.9216


**Step by step:**

1. `PERCENT_RANK()` is `(rank - 1) / (n - 1)`: the top customer, rank 1 out of 51, gets exactly `1.0`; the
   bottom customer would get exactly `0.0`. It answers "what fraction of the population ranks below me".
2. `CUME_DIST()` is `rows at or below this value / n`: the top customer here still gets `1.0`, because with no
   tie above them every row is "at or below" them — but unlike `PERCENT_RANK`, a genuine bottom row gets
   `1 / n`, never `0`, because it is always at or below itself.
3. Both ignore `PARTITION BY` in this query, so they rank across the whole population — add a `PARTITION BY`
   the same way `NTILE` used one, to percentile within a group (spend percentile *within each city*, say).
4. Where `NTILE(4)` could only tell you "top quartile", `PERCENT_RANK` tells you this customer is exactly
   ahead of 96% of the rest — useful the moment a report has to say "top 5%" or "top 1%" and four buckets are
   too coarse to express that.
5. Neither function is an aggregate — like every other window function this level and the last one has used,
   they run after the `WHERE`/`GROUP BY` and do not collapse rows, they annotate them.


## 9. Same Question, Three Tools — Correlated Subquery, Join, or Window

"What was each customer's most recent order date?" has three completely different correct answers in SQL, and
by this point you have met all three. Seeing them side by side, on the same question, is what turns "I know
three ways to do this" into "I know which one to reach for".


In [15]:
correlated = q("""
SELECT c.customer_id, c.name,
       (SELECT o.order_date FROM orders o WHERE o.customer_id = c.customer_id
        ORDER BY o.order_date DESC, o.order_id DESC LIMIT 1) AS last_order_date
FROM customers c
ORDER BY c.customer_id
""")

joined = q("""
WITH last_date AS (
    SELECT customer_id, MAX(order_date) AS last_order_date
    FROM orders GROUP BY customer_id
)
SELECT c.customer_id, c.name, ld.last_order_date
FROM customers c
LEFT JOIN last_date ld ON ld.customer_id = c.customer_id
ORDER BY c.customer_id
""")

windowed = q("""
WITH ranked AS (
    SELECT customer_id, order_date,
           ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date DESC, order_id DESC) AS rn
    FROM orders
)
SELECT c.customer_id, c.name, r.order_date AS last_order_date
FROM customers c
LEFT JOIN ranked r ON r.customer_id = c.customer_id AND r.rn = 1
ORDER BY c.customer_id
""")

print("all three agree:", correlated.equals(joined) and joined.equals(windowed))
correlated.head(5)


all three agree: True


,customer_id,name,last_order_date
0,1,Nisha Joshi,2024-09-21
1,2,Manoj Menon,2024-11-19
2,3,Mohit Pillai,2024-06-03
3,4,Pooja Singh,None
4,5,Varun Chopra,None


**Step by step:**

1. The **correlated subquery** re-runs once per customer, `ORDER BY ... DESC LIMIT 1` picking the latest row
   each time. It reads the most like the English question, and it is the right choice for a one-off, or when
   only a handful of rows need the answer.
2. The **join** pre-aggregates every customer's max date once in a CTE, then joins it on — one pass over
   `orders` instead of one subquery execution per customer. This is usually the fastest of the three once the
   table is large, and it is the shape `sql-intermediate` reached for by default.
3. The **window function** ranks every order within its customer and keeps `rn = 1` — this is the only one of
   the three that generalises for free: swap `LIMIT 1` for `rn <= 3` and you have the top-N-per-group query
   from `sql-intermediate` section 13, with no change to the join or subquery structure needed.
4. All three return the identical DataFrame here, `NULL` and all, for the customers who never ordered — the
   `.equals()` check above is exactly the kind of thing worth doing when you are not sure you have written an
   equivalent query, not just a similar-looking one.
5. Rule of thumb: reach for the correlated subquery when the question really is "one value per row and I only
   need it once"; reach for the join when you will reuse the aggregate or the table is large; reach for the
   window function the moment "most recent" might become "most recent N", because that is a free upgrade.


## 10. A Date Spine Without Recursion

A report with "one row per day" or "one row per month" is wrong the moment a day or month has zero activity —
without a row to hold the zero, that period just does not appear, and a missing bar looks like missing data
rather than an honest zero. `sql-intermediate` section 17 dodged this by using `CASE WHEN` inside an existing
`GROUP BY month`. Building the calendar **first**, independent of what happened, is the more general fix.

`sql-advanced` builds an arbitrary-length date spine with a recursive CTE. For a short, fixed range you know in
advance, a manually written list of offsets is simpler and needs nothing beyond `UNION ALL`.


In [ ]:
q("""
WITH days AS (
    SELECT date('2023-01-01', '+0 days')  AS d UNION ALL SELECT date('2023-01-01', '+1 days')
    UNION ALL SELECT date('2023-01-01', '+2 days') UNION ALL SELECT date('2023-01-01', '+3 days')
    UNION ALL SELECT date('2023-01-01', '+4 days') UNION ALL SELECT date('2023-01-01', '+5 days')
    UNION ALL SELECT date('2023-01-01', '+6 days') UNION ALL SELECT date('2023-01-01', '+7 days')
    UNION ALL SELECT date('2023-01-01', '+8 days') UNION ALL SELECT date('2023-01-01', '+9 days')
    UNION ALL SELECT date('2023-01-01', '+10 days') UNION ALL SELECT date('2023-01-01', '+11 days')
    UNION ALL SELECT date('2023-01-01', '+12 days') UNION ALL SELECT date('2023-01-01', '+13 days')
)
SELECT d.d AS order_date, COUNT(o.order_id) AS orders
FROM days d
LEFT JOIN orders o ON o.order_date = d.d
GROUP BY d.d
ORDER BY d.d
""")


DatabaseError: Execution failed on sql '
# WITH days AS (
    SELECT date('2023-01-01', '+0 days')  AS d UNION ALL SELECT date('2023-01-01', '+1 days')
    UNION ALL SELECT date('2023-01-01', '+2 days') UNION ALL SELECT date('2023-01-01', '+3 days')
    UNION ALL SELECT date('2023-01-01', '+4 days') UNION ALL SELECT date('2023-01-01', '+5 days')
    UNION ALL SELECT date('2023-01-01', '+6 days') UNION ALL SELECT date('2023-01-01', '+7 days')
    UNION ALL SELECT date('2023-01-01', '+8 days') UNION ALL SELECT date('2023-01-01', '+9 days')
    UNION ALL SELECT date('2023-01-01', '+10 days') UNION ALL SELECT date('2023-01-01', '+11 days')
    UNION ALL SELECT date('2023-01-01', '+12 days') UNION ALL SELECT date('2023-01-01', '+13 days')
# )
# SELECT d.d AS order_date, COUNT(o.order_id) AS orders
# FROM days d
# LEFT JOIN orders o ON o.order_date = d.d
# GROUP BY d.d
# ORDER BY d.d
': unrecognized token: "#"

**Step by step:**

1. `days` is a CTE with no reference to `orders` at all — fourteen literal dates, one `SELECT` per day,
   stacked with `UNION ALL`. It exists purely to be joined against; SQLite has no built-in "generate a series
   of dates" function the way PostgreSQL's `generate_series` does.
2. `LEFT JOIN` from `days` to `orders`, not the other way round, is what guarantees every day survives even
   when it matches nothing — the same `LEFT JOIN ... IS NULL` logic `sql-basics` and `sql-intermediate` used
   for customers and products, applied to dates instead.
3. The shop's very first order is on `2023-01-10` — nine days into the range with zero rows, and this query is
   the only way any of that appears at all. A plain `GROUP BY order_date` on `orders` alone would start the
   report on the 10th and never mention the missing nine days.
4. Fourteen `UNION ALL SELECT` lines does not scale — this technique is for a short, known range: a week, a
   month, a fixed reporting window. The moment the range is a parameter rather than a constant, or spans
   years, write the recursive version instead.
5. The same spine-then-`LEFT JOIN` shape works for any "grid that should be complete" report: months × channels,
   categories × regions, or the date range here — it is the same idea as the `CROSS JOIN` `sql-intermediate`
   section 2 used for a complete grid, just built from a list of dates instead of a second table.


## 11. Refactoring a Subquery Monster Into a CTE Chain

`sql-intermediate` section 11 introduced CTE chains for queries you write from scratch. This section is about
a query you did not write — one nested three subqueries deep, that a teammate needs changed, and that you
would rather rebuild as a chain than debug in place. The technique is: name each nested subquery, pull it out
as its own CTE, and confirm the answer has not moved.

The question: which customers, in cities with more than one customer, out-spend the **average customer in
their own city**?


In [17]:
nested = q("""
SELECT c.name, c.city,
       (SELECT ROUND(SUM(i.quantity * i.unit_price * (1 - i.discount)), 2)
        FROM orders o JOIN order_items i ON i.order_id = o.order_id
        WHERE o.customer_id = c.customer_id AND o.status != 'cancelled') AS ltv
FROM customers c
WHERE c.city IS NOT NULL
  AND (SELECT SUM(i.quantity * i.unit_price * (1 - i.discount))
       FROM orders o JOIN order_items i ON i.order_id = o.order_id
       WHERE o.customer_id = c.customer_id AND o.status != 'cancelled') >
      (SELECT AVG(city_ltv) FROM (
          SELECT SUM(i2.quantity * i2.unit_price * (1 - i2.discount)) AS city_ltv
          FROM customers c2
          JOIN orders o2 ON o2.customer_id = c2.customer_id
          JOIN order_items i2 ON i2.order_id = o2.order_id
          WHERE c2.city = c.city AND o2.status != 'cancelled'
          GROUP BY c2.customer_id
      ))
ORDER BY ltv DESC
""")


In [18]:
chained = q("""
WITH customer_ltv AS (          -- one row per customer, their lifetime spend
    SELECT c.customer_id, c.name, c.city,
           SUM(i.quantity * i.unit_price * (1 - i.discount)) AS ltv
    FROM customers c
    JOIN orders o ON o.customer_id = c.customer_id AND o.status != 'cancelled'
    JOIN order_items i ON i.order_id = o.order_id
    WHERE c.city IS NOT NULL
    GROUP BY c.customer_id
),
city_avg AS (                   -- one row per city, its average customer spend
    SELECT city, AVG(ltv) AS avg_ltv
    FROM customer_ltv
    GROUP BY city
)
SELECT cl.name, cl.city, ROUND(cl.ltv, 2) AS ltv
FROM customer_ltv cl
JOIN city_avg ca ON ca.city = cl.city
WHERE cl.ltv > ca.avg_ltv
ORDER BY cl.ltv DESC
""")

print("same answer:", nested.equals(chained))
chained.head(5)


same answer: True


,name,city,ltv
0,Hema Khan,Mumbai,2835360.0
1,Neha Reddy,Mumbai,1539182.5
2,Zara Mehta,Hyderabad,1257470.0
3,Parvati Chopra,Kochi,826850.0
4,Yash Bose,Kochi,791490.0


**Step by step:**

1. The nested version has three subqueries computing overlapping work: a customer's own `ltv` is computed
   *twice* — once for display, once for the `WHERE` comparison — and the city average recomputes every other
   customer's `ltv` in that city, for every row. It is correct, and unreadable, and slow in a way `EXPLAIN`
   would confirm in `sql-advanced`.
2. `customer_ltv` names the first repeated idea — "one row per customer, their spend" — as its own step, with
   a comment stating its grain, the habit `sql-intermediate` section 23 recommended.
3. `city_avg` is the second repeated idea, built **on top of** `customer_ltv` instead of recomputing raw
   `orders`/`order_items` again — that reuse is the entire performance win, and it falls out naturally once the
   first step has a name.
4. The final `SELECT` is now a plain two-table join with a `WHERE`, because both comparisons the nested version
   buried in subqueries are now ordinary columns on ordinary tables.
5. `nested.equals(chained)` is the check that refactoring actually preserved the answer — the whole point of a
   refactor is that nothing observable changes. Run a check like this any time you rewrite a query somebody
   already relies on, before you replace it.


## 12. Three Classic Interview Patterns

Three shapes that show up in SQL interviews often enough to be worth having ready, none of which need
anything beyond what this course has already covered.


In [19]:
# 1. second-highest price, without LIMIT/OFFSET
q("""
SELECT MAX(price) AS second_highest_price
FROM products
WHERE price < (SELECT MAX(price) FROM products)
""")


,second_highest_price
0,118000.0


In [20]:
# 2. the longest gap, in days, between a customer's consecutive orders
q("""
WITH ordered AS (
    SELECT customer_id, order_date,
           LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date, order_id) AS prev_date
    FROM orders WHERE status != 'cancelled'
),
gaps AS (
    SELECT customer_id,
           CAST(julianday(order_date) - julianday(prev_date) AS INTEGER) AS gap_days
    FROM ordered WHERE prev_date IS NOT NULL
)
SELECT customer_id, MAX(gap_days) AS longest_gap_days
FROM gaps
GROUP BY customer_id
ORDER BY longest_gap_days DESC, customer_id
LIMIT 5
""")


,customer_id,longest_gap_days
0,54,691
1,30,526
2,45,455
3,37,386
4,58,323


In [7]:
# 3. this year vs last year, month by month
q("""
WITH monthly AS (
    SELECT strftime('%Y', o.order_date) AS yr, strftime('%m', o.order_date) AS mo,
           SUM(i.quantity * i.unit_price * (1 - i.discount)) AS revenue
    FROM orders o JOIN order_items i ON i.order_id = o.order_id
    WHERE o.status != 'cancelled'
    GROUP BY yr, mo
)
SELECT m24.mo AS month,
       ROUND(m24.revenue, 2) AS revenue_2024,
       ROUND(m23.revenue, 2) AS revenue_2023,
       ROUND(m24.revenue - m23.revenue, 2) AS yoy_change,
       ROUND((m24.revenue - m23.revenue) * 100.0 / m23.revenue, 1) AS yoy_pct
FROM monthly m24
JOIN monthly m23 ON m23.mo = m24.mo AND m23.yr = '2023'
WHERE m24.yr = '2024'
ORDER BY month
LIMIT 5
""")


,month,revenue_2024,revenue_2023,yoy_change,yoy_pct
0,01,414130.0,178950.0,235180.0,131.4
1,02,315950.0,447725.0,-131775.0,-29.4
2,03,983710.0,704500.0,279210.0,39.6
3,04,321250.0,739490.0,-418240.0,-56.6
4,05,1246130.0,442100.0,804030.0,181.9


**Step by step:**

1. **Second-highest, no `LIMIT`/`OFFSET`.** `MAX(price) WHERE price < (SELECT MAX(price) FROM products)` finds
   the largest value that is not *the* largest. It generalises awkwardly (third-highest needs a third nested
   comparison) but it is the answer interviewers usually want, because it works in engines and versions where
   window functions do not exist at all — and it is a scalar subquery you have used all through this course.
   For an N-th-highest that needs to generalise, `DENSE_RANK()` in a CTE with `WHERE rnk = N` is the better
   real-world tool.
2. **Longest gap.** `LAG` fetches each customer's previous order date; `julianday(a) - julianday(b)` converts
   the difference between two `YYYY-MM-DD` text dates into a day count `sql-intermediate` never needed, because
   its date arithmetic stayed at month granularity. `MAX(gap_days)` per customer answers "how close did this
   customer come to churning, at their worst".
3. **Year-over-year.** Two copies of the same `monthly` CTE, joined to each other on the month, one filtered to
   this year and one to last. This is a self-join on a CTE rather than a table — the same mechanism as section
   3's product pairs, applied to time instead of products.
4. All three patterns reuse a tool from earlier in this course or `sql-intermediate` — a scalar subquery,
   `LAG`, a self-join — aimed at a question phrased differently. Interview SQL rarely needs a technique this
   course has not covered; it needs recognising which technique a new-sounding question is actually asking for.
5. The year-over-year join as written silently drops any month that exists in one year but not the other,
   because it is an inner join on `mo`. Section 14 comes back to exactly this.


## 13. If You Move to PostgreSQL or MySQL

Everything new in this level is standard SQL and travels well. The rough edges are narrower than
`sql-intermediate` found, but two of them are worth knowing before you hit them.

| Task | SQLite (here) | PostgreSQL | MySQL 8+ |
| --- | --- | --- | --- |
| `FIRST_VALUE` / `LAST_VALUE` | yes, 3.25+ | yes | yes |
| `PERCENT_RANK` / `CUME_DIST` | yes, 3.25+ | yes | yes |
| `RANGE` frames | yes | yes | yes |
| Day difference between dates | `julianday(a) - julianday(b)` | `a::date - b::date` (integer) | `DATEDIFF(a, b)` |
| Generate a date series | manual `UNION ALL`, or recursive CTE | `generate_series(start, stop, interval)` | manual, or recursive CTE (8.0+) |
| `UNPIVOT` keyword | not available — use `UNION ALL` | not available — use `UNION ALL` | not available — use `UNION ALL` |
| `PIVOT` keyword | not available | not available (use `CASE WHEN` or `crosstab`) | not available |

Two behavioural notes:

- **PostgreSQL's `generate_series`** makes section 10's date spine a one-liner —
  `SELECT generate_series('2023-01-01'::date, '2023-01-14'::date, '1 day')` — which is the main reason the
  manual `UNION ALL` version is worth knowing rather than memorising: it is the fallback for every engine that
  does not have that function, SQLite included.
- **Neither PostgreSQL nor SQL Server-style `PIVOT`/`UNPIVOT` exist in SQLite or MySQL.** The `UNION ALL`
  technique in section 5 is the one spelling that works everywhere, which is exactly why this course teaches
  that version instead of a vendor keyword.


In [22]:
print("SQLite:", q("SELECT sqlite_version() AS v")["v"][0])

# the portable day-difference calculation, in case julianday feels unfamiliar
q("""
SELECT order_id, order_date,
       CAST(julianday(order_date) - julianday('2023-01-01') AS INTEGER) AS days_since_start
FROM orders
ORDER BY order_date
LIMIT 3
""")


SQLite: 3.51.0


,order_id,order_date,days_since_start
0,1,2023-01-10,9
1,4,2023-01-10,9
2,3,2023-01-26,25


**Step by step:**

1. `julianday(d)` converts a date to a floating-point day count since a fixed epoch; subtracting two of them
   gives a day difference directly, which is what section 12's longest-gap query relied on.
2. Casting to `INTEGER` drops any fractional part. SQLite's dates are date-only in this schema (no time
   component), so the subtraction is already a whole number and the cast is just making that explicit.
3. `DATEDIFF` in MySQL and date subtraction in PostgreSQL do the same job with less ceremony — `julianday` is
   SQLite's way of avoiding a dedicated date-arithmetic function set.
4. None of this level's window functions (`FIRST_VALUE`, `LAST_VALUE`, `PERCENT_RANK`, `CUME_DIST`) need a
   version check beyond what `sql-intermediate` already required — SQLite 3.25 covers everything in both
   levels.
5. If a query in this notebook fails to parse on an older SQLite build, `sqlite_version()` is the first thing
   to check, exactly as `sql-intermediate` section 22 advised.


## 14. This Level's Pitfalls

Five mistakes this level makes newly possible, each one silent rather than an error.

| Pitfall | What happens | Fix |
| --- | --- | --- |
| `LAST_VALUE` with no explicit frame | returns the *current* row's value, not the partition's last | add `RANGE`/`ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING` |
| A running total over tied `ORDER BY` values, using `RANGE` | every tied row gets the same, later total | use `ROWS` unless you specifically want peer rows merged |
| Self-join for association without `p2 > p1` (or `<>`) | a product pairs with itself, and every real pair is doubled | add the guard in the `ON` clause |
| Deduplicating without a deterministic tiebreaker in `ORDER BY` | which duplicate survives becomes unpredictable | always end the `ORDER BY` in a unique column |
| An inner join between "this year" and "last year" on month | a month with no orders in either year vanishes from the report instead of showing a gap | `LEFT JOIN` from the side that must always appear |

The last row is section 12's year-over-year query. It happens not to lose any months in this dataset, but
that is luck, not correctness — the query would be wrong the moment a month goes quiet.


In [23]:
# proving the year-over-year join is an inner join in disguise
inner_count = q("""
WITH monthly AS (
    SELECT strftime('%Y', order_date) AS yr, strftime('%m', order_date) AS mo, COUNT(*) AS n
    FROM orders GROUP BY yr, mo
)
SELECT COUNT(*) AS months_compared
FROM monthly m24
JOIN monthly m23 ON m23.mo = m24.mo AND m23.yr = '2023'
WHERE m24.yr = '2024'
""")["months_compared"][0]

left_count = q("""
WITH monthly AS (
    SELECT strftime('%Y', order_date) AS yr, strftime('%m', order_date) AS mo, COUNT(*) AS n
    FROM orders GROUP BY yr, mo
)
SELECT COUNT(*) AS months_compared
FROM monthly m24
LEFT JOIN monthly m23 ON m23.mo = m24.mo AND m23.yr = '2023'
WHERE m24.yr = '2024'
""")["months_compared"][0]

print("INNER JOIN version:", inner_count, "months")
print("LEFT  JOIN version:", left_count, "months")


INNER JOIN version: 12 months
LEFT  JOIN version: 12 months


**Step by step:**

1. Both counts come out at 12 here, because every month in the two-year range has at least one order — this
   dataset never actually hits the bug. That is precisely the danger: the inner-join version *looks* identical
   to the safe version until the day a month is genuinely empty, and then it silently drops a row instead of
   raising an error.
2. `LEFT JOIN` from `m24` guarantees every 2024 month appears in the output even if 2023 has no matching month,
   with `revenue_2023` coming back `NULL` rather than the row disappearing — the same principle `sql-basics`
   taught for `LEFT JOIN ... IS NULL` and `sql-intermediate` taught for anti-joins, applied to a self-join on
   time.
3. The general rule for all five pitfalls above: every one of them is a case where SQL happily returns *a*
   number, and the number is defensible-looking, and it is still not the number the question was actually
   asking for.
4. None of these are things `sql-basics` or `sql-intermediate` could have taught, because none of `LAST_VALUE`,
   `RANGE` frames, association self-joins, deliberate deduplication, or multi-year joins existed yet in those
   levels' vocabulary.
5. `sql-advanced` picks up from here with a different category of mistake entirely: queries that are correct
   but slow, which is a question `EXPLAIN` answers and nothing in this section can.


## Cheat Sheet

| Task | SQL |
| --- | --- |
| Keep the first of each duplicate group | `ROW_NUMBER() OVER (PARTITION BY key ORDER BY tiebreaker) ... WHERE rn = 1` |
| Pairs that co-occur in a group | self-join on the group key with `t2.id > t1.id` (or `<>` for both directions) |
| Flatten N known levels of a hierarchy | chain N self-`LEFT JOIN`s, one per level |
| Unpivot columns to rows | `SELECT id, 'col_a' AS metric, col_a AS value FROM t UNION ALL SELECT id, 'col_b', col_b FROM t` |
| A group's first/last value on every row | `FIRST_VALUE(x) OVER (PARTITION BY g ORDER BY o)` / `LAST_VALUE(x) OVER (... RANGE BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING)` |
| Running total that respects ties correctly | `SUM(x) OVER (ORDER BY o ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)` |
| Exact percentile of every row | `PERCENT_RANK() OVER (ORDER BY x)` / `CUME_DIST() OVER (ORDER BY x)` |
| Same value, three ways | correlated subquery (one-off) / join to a pre-aggregate (reused, large table) / window function (might need top-N later) |
| A calendar with no data-driven gaps | `WITH days AS (SELECT date1 UNION ALL SELECT date2 ...) SELECT ... FROM days LEFT JOIN t ON ...` |
| Refactor nested subqueries | name each nested `SELECT` as its own CTE, built on the one before it |
| Second-highest without `LIMIT`/`OFFSET` | `MAX(x) WHERE x < (SELECT MAX(x) FROM t)` |
| Day difference between two dates | `julianday(a) - julianday(b)` |
| This period vs a year ago | self-join two copies of the same monthly CTE on the period, `LEFT JOIN` from the side that must appear |

## Suggested Learning Path

1. Sections 2 and 3 are both self-joins doing unrelated jobs — dedup and association. Notice what changes in
   the `ON` clause between them and what stays the same.
2. Section 4 is a bridge to `sql-advanced`: write it by hand here, then go compare it against that level's
   recursive CTE once you get there, on the same `employees` table.
3. Do sections 6 and 7 back to back. `LAST_VALUE`'s default-frame trap and the `RANGE`/`ROWS` tie behaviour
   are two faces of the same idea — the frame decides which rows are "in scope" for a window function, and
   the default is not always the one you want.
4. Section 9 is a checkpoint, not new material — if any of the three queries there feels unfamiliar, that is
   the level to revisit, not this one.
5. Sections 10 and 11 are both about restructuring a query without changing its answer. Run the `.equals()` /
   count checks yourself rather than trusting that the rewrite worked.
6. Section 12's three patterns are worth being able to write from a blank cell, no guide open.
7. Finish on section 14 and check you can explain all five pitfalls, especially the `LAST_VALUE` one — it is
   the one most likely to reappear in your own work without a lesson attached to it.

## Where to Go Next

- **`sql-advanced`** picks up immediately: query plans, indexes, the recursive CTE that generalises section 4
  and section 10, transactions in earnest, and schema design.
- Take the market-basket query from section 3 and run it against a real e-commerce export if you have access
  to one. The self-join scales fine; the interesting part is what the actual pairs turn out to be.
- Revisit any report you built in `sql-intermediate` that has a chart with missing bars, and fix it with a
  spine like section 10's.

## Practice Next

Open `sql-intermediate-2-exercises.ipynb`. It follows this guide section by section, and ends with two mini
projects that combine several of this level's techniques into one report each — keep them, the way
`sql-intermediate`'s cohort table and best-seller report were worth keeping.
